In [10]:
!apt-get update -qq
!apt-get install -y nvidia-cuda-toolkit -q
!pip install pycuda streamlit pyngrok opencv-python-headless --quiet


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: dpkg was interrupted, you must manually run 'dpkg --configure -a' to correct the problem. 


In [11]:
!nvidia-smi

Mon Oct 27 14:00:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!ngrok config add-authtoken 33ps38vcPItBfai7RTicbZVj1Uc_28Anx5RcpnxWzJ5jDJ3cg

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [13]:
'''%%writefile app.py
import streamlit as st
import cv2
import numpy as np
import pycuda.autoinit
import pycuda.driver as drv
from pycuda.compiler import SourceModule

st.title("GPU-Accelerated Edge Detection")

uploaded_file = st.file_uploader("Upload a video file", type=["mp4", "avi", "mov"])
stframe = st.empty()

if uploaded_file is not None:
    # Save uploaded video
    with open("/tmp/input_video.mp4", "wb") as f:
        f.write(uploaded_file.read())

    cap = cv2.VideoCapture("/tmp/input_video.mp4")
    ret, frame = cap.read()
    if not ret:
        st.error("Error: Could not read the video")
        st.stop()

    height, width = frame.shape[:2]

    # Sobel kernel in PyCUDA
    sobel_kernel = """
    __global__ void sobel_filter(unsigned char *gray_img, unsigned char *output_img, int width, int height){
        int x = blockIdx.x * blockDim.x + threadIdx.x;
        int y = blockIdx.y * blockDim.y + threadIdx.y;
        if (x >= width || y >= height) return;
        int gx=0, gy=0;
        int sobel_x[3][3] = {{-1,0,1},{-2,0,2},{-1,0,1}};
        int sobel_y[3][3] = {{-1,-2,-1},{0,0,0},{1,2,1}};
        for(int i=-1;i<=1;i++){
            for(int j=-1;j<=1;j++){
                int xi = min(max(x+i,0), width-1);
                int yj = min(max(y+j,0), height-1);
                int val = gray_img[yj*width+xi];
                gx += sobel_x[i+1][j+1]*val;
                gy += sobel_y[i+1][j+1]*val;
            }
        }
        int mag = sqrtf(float(gx*gx + gy*gy));
        output_img[y*width+x] = mag>255?255:mag;
    }
    """

    mod = SourceModule(sobel_kernel)
    sobel_filter = mod.get_function("sobel_filter")

    block_size = (16,16,1)
    grid_size = (int(np.ceil(width/16)), int(np.ceil(height/16)),1)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        input_gpu = drv.mem_alloc(gray.nbytes)
        output_gpu = drv.mem_alloc(gray.nbytes)
        drv.memcpy_htod(input_gpu, gray)
        output = np.zeros_like(gray)

        sobel_filter(input_gpu, output_gpu, np.int32(width), np.int32(height),
                     block=block_size, grid=grid_size)

        drv.memcpy_dtoh(output, output_gpu)
        stframe.image(output, channels="GRAY")

    cap.release()'''

'%%writefile app.py\nimport streamlit as st\nimport cv2\nimport numpy as np\nimport pycuda.autoinit\nimport pycuda.driver as drv\nfrom pycuda.compiler import SourceModule\n\nst.title("GPU-Accelerated Edge Detection")\n\nuploaded_file = st.file_uploader("Upload a video file", type=["mp4", "avi", "mov"])\nstframe = st.empty()\n\nif uploaded_file is not None:\n    # Save uploaded video\n    with open("/tmp/input_video.mp4", "wb") as f:\n        f.write(uploaded_file.read())\n\n    cap = cv2.VideoCapture("/tmp/input_video.mp4")\n    ret, frame = cap.read()\n    if not ret:\n        st.error("Error: Could not read the video")\n        st.stop()\n\n    height, width = frame.shape[:2]\n\n    # Sobel kernel in PyCUDA\n    sobel_kernel = """\n    __global__ void sobel_filter(unsigned char *gray_img, unsigned char *output_img, int width, int height){\n        int x = blockIdx.x * blockDim.x + threadIdx.x;\n        int y = blockIdx.y * blockDim.y + threadIdx.y;\n        if (x >= width || y >= hei

In [14]:
'''# ===============================
# STEP 5: Launch Streamlit + Ngrok
# ===============================
import subprocess
import time
from pyngrok import ngrok

# Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.address", "0.0.0.0",
                  "--server.headless", "true"])

# Wait for Streamlit to start
time.sleep(15)

# Connect Ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app is live at: {public_url}")'''


'# ===============================\n# STEP 5: Launch Streamlit + Ngrok\n# ===============================\nimport subprocess\nimport time\nfrom pyngrok import ngrok\n\n# Start Streamlit in background\nsubprocess.Popen(["streamlit", "run", "app.py",\n                  "--server.port", "8501",\n                  "--server.address", "0.0.0.0",\n                  "--server.headless", "true"])\n\n# Wait for Streamlit to start\ntime.sleep(15)\n\n# Connect Ngrok tunnel\npublic_url = ngrok.connect(8501)\nprint(f"Streamlit app is live at: {public_url}")'

In [15]:
# Install dependencies
!pip install pycuda streamlit pyngrok opencv-python-headless --quiet

# Ngrok token
!ngrok config add-authtoken YOUR_NGROK_AUTHTOKEN


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [16]:
# Write Streamlit app
%%writefile app.py
import streamlit as st
import cv2
import numpy as np

st.title("Simple Edge Detection")

uploaded_file = st.file_uploader("Upload a video", type=["mp4","avi","mov"])
stframe = st.empty()

if uploaded_file is not None:
    with open("/tmp/input.mp4","wb") as f:
        f.write(uploaded_file.read())

    cap = cv2.VideoCapture("/tmp/input.mp4")
    ret, frame = cap.read()
    if not ret:
        st.error("Cannot read video")
        st.stop()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 100, 200)
        stframe.image(edges, channels="GRAY")
    cap.release()


Overwriting app.py


In [17]:
# Launch Streamlit in background
import subprocess
import time
from pyngrok import ngrok

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.address", "0.0.0.0",
                  "--server.headless", "true"])

# Wait for Streamlit to start properly
time.sleep(20)  # Increase if needed

# Start ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit is live at: {public_url}")


Streamlit is live at: NgrokTunnel: "https://unstrained-sprayfully-davina.ngrok-free.dev" -> "http://localhost:8501"
